In [ ]:
from constants import *
from astropy import units as u

from phoenix_grid_creator.spectral_grid import spectral_grid
from spectrum_component_analyser.phoenix_spectrum import phoenix_spectrum

fits_file_paths = list(Path(package_path / "raw_phoenix_spectra").rglob("*.fits"))
# fits_file_paths =fits_file_paths[:1]
resolution : u.Quantity[u.K] = .01*u.um
spec_grid : spectral_grid = spectral_grid.from_local_raw(fits_file_paths, resolution, False) # need to downsample resolution otherwise the spectral grid won't fit in memory...

# units are getting lost somewhere, idk why
spec_grid.Wavelengths = spec_grid.Wavelengths.to(u.um)
spec_grid.Fluxes *= u.Jy

In [ ]:
spec_grid.get_spectrum(spec_grid.T_effs[-1], spec_grid.FeHs[0], spec_grid.Log_gs[0]).plot()

In [ ]:
%matplotlib inline
# take 2 random spectra
# combine them with random weights
# see if you can reach a global minimum at a given resolution
# repeat for many resolutions

number_of_components : int = 2

from matplotlib import pyplot as plt

import rich
from spectrum_component_analyser.mcmc.simulated_spectra import get_random_simulated_spectrum, get_simulated_spectra
from spectrum_component_analyser.spectral_component import spectral_component
from spectrum_component_analyser.spectrum import spectrum

true_components, component_spectra, combined_spectrum = get_random_simulated_spectrum(number_of_components, spec_grid, FeH=0.0 * u.dex, log_g=4.5 * u.dex)

comps = [
    spectral_component(3800 * u.K, 0.0 *  u.dex, 4.5 * u.dex, 0.85),
    spectral_component(3300 * u.K, 0.0 *  u.dex, 4.5 * u.dex, 0.15)
]

true_components, component_spectra, combined_spectrum = get_simulated_spectra(spec_grid, comps)

true_table = spectral_component.return_default_table()
for c in true_components:
    c.pretty_print(true_table)
rich.print(true_table)

In [ ]:
"""
https://www.aanda.org/articles/aa/pdf/2016/03/aa22261-13.pdf talks about a dynamical mask, could maybe introduce that?

e.g. using

total_fluxes : np.ndarray = np.zeros((len(spec_grid.Wavelengths))) * u.Jy

all_phoenix_params = list(itertools.product(spec_grid.T_effs, spec_grid.FeHs, spec_grid.Log_gs))

for t, f, l in all_phoenix_params:
    total_fluxes += spec_grid.LookupTable[t, f, l]

average_flux = total_fluxes / len(all_phoenix_params)

average_flux = average_flux[mask]
"""

import numpy as np
import scipy as sp
from spectrum_component_analyser.interpolated_spectrum import get_interpolated_phoenix_spectrum
from spectrum_component_analyser.chi_squared_minimisation import ChiHelper

number_of_parameters : int = 4 # weight, t, f, l

chi : ChiHelper = ChiHelper(
    spec_grid=spec_grid,
    number_of_components=number_of_components,
    number_of_parameters=number_of_parameters,
    observed_spectrum=combined_spectrum
)

parameter_bounds = [
        (.0, 2.),
        (np.min(spec_grid.T_effs.value), np.max(spec_grid.T_effs.value)),
        (np.min(spec_grid.FeHs.value), np.max(spec_grid.FeHs.value)),
        # (target.feh.value * .9, target.feh.value * 1.1),
        # (0, 0),
        (np.min(spec_grid.Log_gs.value), np.max(spec_grid.Log_gs.value)),
        # (target.log_g.value * .9, target.log_g.value * 1.1)
        # (1.9, 2.1),
    ]

r = chi.get_r(parameter_bounds)

print(r)

In [ ]:
%matplotlib inline

print("simulated / true")
rich.print(true_table)

chi.plot(r)

In [ ]:
from spectrum_component_analyser.mcmc.helper import MCMCHelper

mcmc : MCMCHelper = MCMCHelper(
    parameter_bounds=parameter_bounds,
    number_of_components=number_of_components,
    number_of_parameters=number_of_parameters,
    observed_spectrum=combined_spectrum,
    spec_grid=spec_grid,
    n_steps = 5000,
    n_walkers=32
)

sampler, samples = mcmc.run(r)


In [ ]:
print("\n[ORIGINAL PARAMETERS]")
rich.print(true_table)

mcmc.print_parameters(samples=samples)

In [ ]:
%matplotlib inline

print(true_components[0].Log_g.value)
mcmc.plot_corner(sampler, samples, true_components)

In [ ]:
print("\n[ORIGINAL PARAMETERS]")
rich.print(true_table)

mcmc.print_parameters(samples=samples)

In [ ]:
print("\n[ORIGINAL PARAMETERS]")
rich.print(true_table)

mcmc.print_parameters(samples=samples)

In [ ]:
%matplotlib widget
mcmc.plot_spectrum(samples)